# A5B: CNN Experiments - Pipeline de Experimentação

Notebook para testar diferentes configurações de hiperparâmetros usando `ExperimentRunner`.

Cada experimento é salvo com:
- Modelo treinado (.h5)
- Histórico de treinamento (loss/accuracy)
- Cópia da configuração usada (para rastreabilidade)

## 1. Importações e Setup

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import json

# Adicionar src ao path (relativo - funciona em qualquer lugar)
notebook_dir = Path.cwd()
src_path = notebook_dir / "src" if (notebook_dir / "src").exists() else notebook_dir.parent / "src"
sys.path.insert(0, str(src_path))

# Importar ferramentas de experimentação
from models.experiment_runner import ExperimentRunner, run_multiple_experiments
from models.cnn_config import list_available_configs, load_config

print("✓ Importações realizadas com sucesso!")

✓ Importações realizadas com sucesso!


## 2. Listar Configurações Disponíveis

In [2]:
# Ver todas as configurações disponíveis
available_configs = list_available_configs()

print("\nConfiguções disponíveis para experimentação:\n")
for i, config_name in enumerate(available_configs, 1):
    print(f"  {i}. {config_name}")
    
    # Carregar e exibir parâmetros principais
    config = load_config(config_name)
    model_cfg = config.get("model", {})
    train_cfg = config.get("training", {})
    
    print(f"     Filtros: {model_cfg.get('filters')}")
    print(f"     L2: {model_cfg.get('l2_regularizer')}")
    print(f"     Conv Dropout: {model_cfg.get('conv_dropout_rate')}")
    print(f"     Dense Dropout: {model_cfg.get('dense_dropout_rate')}")
    print(f"     Learning Rate: {train_cfg.get('learning_rate')}")
    print()


Configuções disponíveis para experimentação:

  1. baseline
     Filtros: [32, 64]
     L2: 0.001
     Conv Dropout: 0.2
     Dense Dropout: 0.5
     Learning Rate: 0.001

  2. higher_dropout
     Filtros: [32, 64]
     L2: 0.001
     Conv Dropout: 0.3
     Dense Dropout: 0.6
     Learning Rate: 0.0005



## 3. Rodar Um Experimento

## ⚠️ Avisos Antes de Rodar Experimentos

### Parâmetros Recomendados

| Param | Valor | Descrição |
|-------|-------|-----------|
| `limit_samples` | `10` | ✅ **Teste rápido** (1-2 min) - Use para validação |
| `limit_samples` | `100-150` | ⚡ **Teste médio** (10-15 min) - Bom balanço |
| `limit_samples` | `None` | 🔥 **Dados completos** (30+ min) - Produção final |

### Estimativas de Tempo
- **10 amostras**: ~1-2 minutos
- **100-150 amostras**: ~10-15 minutos  
- **Todos os dados**: ~30-60 minutos (depende do hardware)

### Requisitos
- 💾 **Espaço em disco**: ~500MB por experimento (modelo + histórico)
- 🖥️ **RAM**: 4GB mínimo
- ⚡ **GPU**: Recomendado (TensorFlow usa CPU se não houver GPU)

### Dicas de Uso
1. **Comece pequeno**: Use `limit_samples=10` para testar o workflow
2. **Incremente gradualmente**: Se funcionar, tente `limit_samples=100` e depois `limit_samples=None`
3. **Não interrompa**: Se parar no meio, o resultado NÃO é salvo
4. **Checker histórico**: Section 7 mostra todos os experimentos rodados

### Não Modifique
- ❌ Não altere `baseline.yaml` diretamente (será sobrescrito)
- ❌ Não delete arquivos em `outputs/trained_models/` manualmente
- ✅ Crie **novas configurações** para ablações (Section 8)


In [3]:
# Teste com mais amostras para ablação de dropout
print("\nRodando experimento BASELINE...\n")

runner = ExperimentRunner("baseline")
result_baseline = runner.run_full_pipeline(
    limit_samples=100,  # ABLAÇÃO: 100 (10-15 min) | PRODUÇÃO: None (todos dados)
    verbose=1
)

print(f"\nExperimento salvo em: {result_baseline['experiment_dir']}")



Rodando experimento BASELINE...

✓ ExperimentRunner inicializado com config: baseline
  Configurações disponíveis: ['baseline', 'higher_dropout']

EXPERIMENTO: baseline

📂 Carregando dados...
✓ Dados carregados: (100, 128, 128, 9), labels: (100,)
  Classes: [0 1], distribuição: [59 41]

🏗️ Construindo modelo...
✓ Modelo compilado!

💾 Experimento: baseline_20260309_221622

🚀 Iniciando treinamento...
  Batch size: 32
  Epochs: 50
  Learning rate: 0.001
  Train size: 80, Val size: 20
Epoch 1/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 395ms/step - accuracy: 0.5750 - loss: 6.2019 - val_accuracy: 0.7000 - val_loss: 2.2657
Epoch 2/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 285ms/step - accuracy: 0.7125 - loss: 4.0420 - val_accuracy: 0.5000 - val_loss: 2.2223
Epoch 3/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 319ms/step - accuracy: 0.7625 - loss: 2.2329 - val_accuracy: 0.8000 - val_loss: 1.3914
Epoch 4/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 337ms/step - accuracy: 0.8000 - loss: 1.3627 - val_accuracy: 0.6500 - val_loss: 1.0529
Epoch 5/5

✓ Treinamento concluído!
✓ Modelo salvo: c:\Users\Inteli\Desktop\g01\outputs\trained_models\baseline_20260309_221622\model.h5
✓ Histórico salvo: c:\Users\Inteli\Desktop\g01\outputs\trained_models\baseline_20260309_221622\history.json
✓ Novo log de experimentos criado: c:\Users\Inteli\Desktop\g01\outputs\trained_models\experiments_log.csv

RESUMO DO EXPERIMENTO:
  config_name: baseline
  experiment_dir: c:\Users\Inteli\Desktop\g01\outputs\trained_models\baseline_20260309_221622
  timestamp: 2026-03-09 22:17:16
  final_train_loss: 0.3178
  final_train_acc: 0.9875
  final_val_loss: 1.2867
  final_val_acc: 0.8500
  epochs_run: 50
  val_accuracy: 0.8500
  val_precision: 0.8535
  val_recall: 0.8500
  val_f1: 0.8496
  val_balanced_accuracy: 0.8500
  val_auc_roc: 0.5800
  val_pr_auc: 0.5371
  val_cm_tp: 8
  val_cm_fp: 1
  val_cm_tn: 9
  val_cm_fn: 2
  val_specificity: 0.9000
  val_sensitivity: 0.8000

Experimento salvo em: c:\Users\Inteli\Desktop\g01\outputs\trained_models\baseline_20260309_22

## 4. Rodar Múltiplos Experimentos e Comparar

In [4]:
# Rodar múltiplas configurações de dropout (ablação)
print("\nRodando múltiplos experimentos de regularização (Dropout)...\n")

# Ablação: testar conv_dropout e dense_dropout individualmente
dropout_configs = [
    "baseline",
    "regularizacao_conv_dropout_0_3",      # Conv dropout: 0.2 → 0.3
    "regularizacao_conv_dropout_0_4",      # Conv dropout: 0.2 → 0.4
    "regularizacao_dense_dropout_0_6",     # Dense dropout: 0.5 → 0.6
    "regularizacao_dense_dropout_0_7",     # Dense dropout: 0.5 → 0.7
]

results = run_multiple_experiments(
    dropout_configs,
    limit_samples=100  # ABLAÇÃO: 100 (50-75 min total) | PRODUÇÃO: None (todos dados)
)

print("\nExperimentos concluídos!")



Rodando múltiplos experimentos de regularização (Dropout)...

✓ ExperimentRunner inicializado com config: baseline
  Configurações disponíveis: ['baseline', 'higher_dropout']

EXPERIMENTO: baseline

📂 Carregando dados...
✓ Dados carregados: (100, 128, 128, 9), labels: (100,)
  Classes: [0 1], distribuição: [59 41]

🏗️ Construindo modelo...
✓ Modelo compilado!

💾 Experimento: baseline_20260309_221807

🚀 Iniciando treinamento...
  Batch size: 32
  Epochs: 50
  Learning rate: 0.001
  Train size: 80, Val size: 20
Epoch 1/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 474ms/step - accuracy: 0.5000 - loss: 11.6176 - val_accuracy: 0.5000 - val_loss: 1.5758
Epoch 2/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 316ms/step - accuracy: 0.6500 - loss: 2.9963 - val_accuracy: 0.7500 - val_loss: 1.0625
Epoch 3/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 330ms/step - accuracy: 0.7500 - loss: 1.5994 - val_accuracy: 0.8500 - val_loss: 0.8328
Epoch 4/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 349ms/step - accuracy: 0.8375 - loss: 0.8270 - val_accuracy: 0.700

✓ Treinamento concluído!
✓ Modelo salvo: c:\Users\Inteli\Desktop\g01\outputs\trained_models\baseline_20260309_221807\model.h5
✓ Histórico salvo: c:\Users\Inteli\Desktop\g01\outputs\trained_models\baseline_20260309_221807\history.json
✓ Resultado adicionado ao log: c:\Users\Inteli\Desktop\g01\outputs\trained_models\experiments_log.csv

RESUMO DO EXPERIMENTO:
  config_name: baseline
  experiment_dir: c:\Users\Inteli\Desktop\g01\outputs\trained_models\baseline_20260309_221807
  timestamp: 2026-03-09 22:19:01
  final_train_loss: 0.3334
  final_train_acc: 0.9875
  final_val_loss: 1.3967
  final_val_acc: 0.8500
  epochs_run: 50
  val_accuracy: 0.8500
  val_precision: 0.8535
  val_recall: 0.8500
  val_f1: 0.8496
  val_balanced_accuracy: 0.8500
  val_auc_roc: 0.1800
  val_pr_auc: 0.4474
  val_cm_tp: 8
  val_cm_fp: 1
  val_cm_tn: 9
  val_cm_fn: 2
  val_specificity: 0.9000
  val_sensitivity: 0.8000
❌ Erro em regularizacao_conv_dropout_0_3: Configuração não encontrada: c:\Users\Inteli\Desktop\g01

## 5. Comparar Resultados

In [5]:
# Criar tabela de comparação
comparison_data = []

for r in results:
    if "error" not in r:
        comparison_data.append({
            "Config": r.get("config_name"),
            "Train Loss": f"{r.get('final_train_loss', 0):.4f}",
            "Train Acc": f"{r.get('final_train_acc', 0):.4f}",
            "Val Loss": f"{r.get('final_val_loss', 0):.4f}",
            "Val Acc": f"{r.get('final_val_acc', 0):.4f}",
            "Epochs": r.get("epochs_run"),
        })
    else:
        print(f"❌ Erro em {r.get('config_name')}: {r.get('error')}")

comparison_df = pd.DataFrame(comparison_data)

print("\n" + "="*100)
print("COMPARAÇÃO DE EXPERIMENTOS")
print("="*100)
print(comparison_df.to_string(index=False))
print("="*100)

❌ Erro em regularizacao_conv_dropout_0_3: Configuração não encontrada: c:\Users\Inteli\Desktop\g01\src\models\configs\regularizacao_conv_dropout_0_3.yaml
❌ Erro em regularizacao_conv_dropout_0_4: Configuração não encontrada: c:\Users\Inteli\Desktop\g01\src\models\configs\regularizacao_conv_dropout_0_4.yaml
❌ Erro em regularizacao_dense_dropout_0_6: Configuração não encontrada: c:\Users\Inteli\Desktop\g01\src\models\configs\regularizacao_dense_dropout_0_6.yaml
❌ Erro em regularizacao_dense_dropout_0_7: Configuração não encontrada: c:\Users\Inteli\Desktop\g01\src\models\configs\regularizacao_dense_dropout_0_7.yaml

COMPARAÇÃO DE EXPERIMENTOS
  Config Train Loss Train Acc Val Loss Val Acc  Epochs
baseline     0.3334    0.9875   1.3967  0.8500      50


## 6. Inspecionar Resultados de um Experimento

In [6]:
# Carregar histórico de um experimento
import json

# Usar o caminho do primeiro experimento resultante
if results and "experiment_dir" in results[0]:
    exp_dir = Path(results[0]["experiment_dir"])
    
    # Carregar config usada
    config_file = exp_dir / "config_used.json"
    if config_file.exists():
        with open(config_file) as f:
            config_used = json.load(f)
        print(f"\nConfig usada em {exp_dir.name}:")
        print(json.dumps(config_used, indent=2))
    
    # Carregar histórico
    history_file = exp_dir / "history.json"
    if history_file.exists():
        with open(history_file) as f:
            history = json.load(f)
        
        print(f"\nHistórico de treinamento (últimas 5 épocas):")
        epochs = len(history['loss'])
        for i in range(max(0, epochs-5), epochs):
            print(f"  Época {i+1}: Loss={history['loss'][i]:.4f}, Acc={history['accuracy'][i]:.4f}, "
                  f"Val Loss={history['val_loss'][i]:.4f}, Val Acc={history['val_accuracy'][i]:.4f}")


Config usada em baseline_20260309_221807:
{
  "model": {
    "input_shape": [
      128,
      128,
      9
    ],
    "num_classes": 2,
    "filters": [
      32,
      64
    ],
    "kernel_size": 3,
    "pool_size": 2,
    "l2_regularizer": 0.001,
    "conv_dropout_rate": 0.2,
    "dense_dropout_rate": 0.5,
    "dense_units": 128
  },
  "training": {
    "batch_size": 32,
    "epochs": 50,
    "learning_rate": 0.001,
    "validation_split": 0.2,
    "optimizer": "adam"
  },
  "data": {
    "dataset_path": "c:\\Users\\Inteli\\Desktop\\g01\\outputs\\pixels_dataset.csv",
    "codes_path": "c:\\Users\\Inteli\\Desktop\\g01\\data\\extracted_codes.json",
    "normalizer_path": "c:\\Users\\Inteli\\Desktop\\g01\\outputs\\a04_cnn_data_prep\\cnn_normalizer_zscore.npz",
    "normalization_method": "zscore"
  },
  "output": {
    "models_dir": "c:\\Users\\Inteli\\Desktop\\g01\\outputs\\trained_models",
    "logs_dir": "c:\\Users\\Inteli\\Desktop\\g01\\outputs\\training_logs",
    "save_model": 

## 7. Histórico de Todos os Experimentos

Visualize o arquivo CSV com todos os experimentos rodados (append automático).

In [7]:
# Carregar e exibir histórico de experimentos
# Usar o mesmo caminho que experiment_runner usa
config = load_config("baseline")
output_cfg = config["output"]
csv_path = Path(output_cfg["models_dir"]) / "experiments_log.csv"

if csv_path.exists():
    df_experiments = pd.read_csv(csv_path)
    
    print("\n" + "="*120)
    print("HISTÓRICO DE TODOS OS EXPERIMENTOS")
    print("="*120)
    print(df_experiments.to_string(index=False))
    print("="*120)
    print(f"\n📊 Total de experimentos: {len(df_experiments)}")
    print(f"📈 Melhor Val Acc: {df_experiments['val_acc'].max():.4f} ({df_experiments.loc[df_experiments['val_acc'].idxmax(), 'config_name']})")
    print(f"📉 Menor Val Loss: {df_experiments['val_loss'].min():.4f} ({df_experiments.loc[df_experiments['val_loss'].idxmin(), 'config_name']})")
else:
    print(f"⚠️ Arquivo de log não encontrado em: {csv_path}")


HISTÓRICO DE TODOS OS EXPERIMENTOS
          timestamp config_name  train_loss  train_acc  val_loss  val_acc  epochs  val_accuracy  val_precision  val_recall   val_f1  val_balanced_accuracy  val_auc_roc  val_pr_auc  val_cm_tp  val_cm_fp  val_cm_tn  val_cm_fn  val_specificity  val_sensitivity  val_cm_string                                                              experiment_dir
2026-03-09 22:17:16    baseline    0.317800     0.9875  1.286709     0.85      50          0.85       0.853535        0.85 0.849624                   0.85         0.58    0.537116          8          1          9          2              0.9              0.8            NaN c:\Users\Inteli\Desktop\g01\outputs\trained_models\baseline_20260309_221622
2026-03-09 22:19:01    baseline    0.333408     0.9875  1.396717     0.85      50          0.85       0.853535        0.85 0.849624                   0.85         0.18    0.447394          8          1          9          2              0.9              0.8         

## 8. Criar Novas Configurações para Ablations

Para testar novos hiperparâmetros (ablação study):

### Passo 1: Criar novo YAML
Arquivo: `src/models/configs/seu_nome.yaml`

```yaml
model:
  input_shape: [128, 128, 9]
  num_classes: 2
  filters: [32, 64]           # ← Mude UM parâmetro por vez
  kernel_size: 3
  pool_size: 2
  l2_regularizer: 0.001
  conv_dropout_rate: 0.2
  dense_dropout_rate: 0.5
  dense_units: 128

training:
  batch_size: 32
  epochs: 50
  learning_rate: 0.001
  validation_split: 0.2
  optimizer: adam

data:
  dataset_path: outputs/pixels_dataset.csv
  codes_path: data/extracted_codes.json
  normalizer_path: outputs/a04_cnn_data_prep/cnn_normalizer_zscore.npz
  normalization_method: zscore

output:
  models_dir: outputs/trained_models/
  logs_dir: outputs/training_logs/
  save_model: true
  save_history: true
```

### Passo 2: Rodar experimento
```python
runner = ExperimentRunner("seu_nome")
runner.run_full_pipeline(limit_samples=50)  # Teste primeiro
```

### Passo 3: Comparar resultados
- Veja Section 7 para tabela com todos os resultados
- Dados salvos em: `outputs/trained_models/experiments_log.csv`

### Dicas de Ablação
- **Mude UM parâmetro por vez** (não vários simultaneamente)
- Comece com `limit_samples=10-50` para testes rápidos
- Só use `limit_samples=None` quando tiver validado o experimento